# Process Faces — corrected MySQL version
Fixes the `Unknown column 'face_id'` error by no longer selecting `face_id` from `captured_snapshots`.


In [ ]:
# %pip install opencv-python deepface mysqlclient python-dotenv tf-keras
import os, sys, time
from pathlib import Path
import MySQLdb
from MySQLdb.cursors import DictCursor
from dotenv import load_dotenv

start_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (start_dir, *start_dir.parents) if (path / 'api_server').is_dir()),
    start_dir,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

from api_server.worker import process_snapshot

In [2]:
db = MySQLdb.connect(
    db=os.getenv('MYSQL_DATABASE', 'smart_sentiment'),
    user=os.getenv('MYSQL_USER', 'root'),
    passwd=os.getenv('MYSQL_PASSWORD', ''),
    host=os.getenv('MYSQL_HOST', '127.0.0.1'),
    port=int(os.getenv('MYSQL_PORT', '3306')),
    charset='utf8mb4',
    autocommit=False,
)
print('Database connected')

Database connected


In [3]:
# Optional: confirm the actual columns in captured_snapshots
cur = db.cursor(DictCursor)
cur.execute('SHOW COLUMNS FROM captured_snapshots')
print([row['Field'] for row in cur.fetchall()])
cur.close()


['id', 'job_id', 'pc_name', 'session_id', 'image_path', 'upload_content_type', 'upload_size_bytes', 'timestamp', 'status', 'emotion', 'confidence', 'emotion_vector', 'processed', 'embedding', 'error_message', 'processed_at', 'branch_id', 'device_id', 'visitor_id']


In [4]:
# Reuse the tested application worker instead of maintaining a second
# emotion model implementation in this notebook. Do not run the RQ worker
# at the same time as this notebook processor.


In [5]:
def process_unprocessed_snapshots():
    cur = None
    try:
        db.ping(True)
        cur = db.cursor(DictCursor)
        cur.execute('''
            SELECT id
            FROM captured_snapshots
            WHERE processed = FALSE AND status = 'queued'
            ORDER BY timestamp ASC
        ''')
        rows = cur.fetchall()
        print(f'Found {len(rows)} unprocessed images')

        for row in rows:
            db_id = row['id']
            try:
                result = process_snapshot(db_id)
                print(f"Snapshot {db_id}: {result['status']} -> {result.get('emotion', result.get('reason'))}")
            except Exception as exc:
                print(f'Snapshot {db_id} failed: {exc.__class__.__name__}')

        return len(rows)
    except Exception as e:
        db.rollback()
        print('Processing error:', e)
        return 0
    finally:
        if cur is not None:
            cur.close()


In [ ]:
# Test one pass first
process_unprocessed_snapshots()


In [7]:
def run_worker(poll_seconds=5):
    print('Face processor started. Interrupt the kernel to stop.')
    while True:
        process_unprocessed_snapshots()
        time.sleep(poll_seconds)

# run_worker()
